# Notebook 03 — Graph Analytics: PageRank & Community Detection

## Imports & Neo4j Connection

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from neo4j import GraphDatabase
import warnings
warnings.filterwarnings('ignore')

# ── Neo4j connection ────────────────────────────────────────────────────────
URI      = "neo4j://127.0.0.1:7687"
USER     = "neo4j"
PASSWORD = "juilee1609"   # ← change to match your docker-compose.yml

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

# Convenience wrapper: run a Cypher query, return a DataFrame
def run_query(cypher: str, params: dict = None) -> pd.DataFrame:
    """Execute a Cypher query and return results as a pandas DataFrame."""
    with driver.session() as session:
        result = session.run(cypher, params or {})
        return pd.DataFrame([r.data() for r in result])

# Verify connection
test = run_query("RETURN 'Connected to Neo4j' AS status")
print(test.iloc[0]['status'])

Connected to Neo4j


In [10]:
result = run_query("""
CALL db.labels() YIELD label
RETURN label
ORDER BY label
""")

print(result)

        label
0    Customer
1  Department
2       Order
3     Product
4      Region


In [11]:
labels = [
    'Customer', 'Department', 'Order',
    'Product', 'Region'
]

#existing = run_query("CALL db.labels() YIELD label RETURN label")['label'].tolist()

counts = {}

for label in labels:
    if label not in existing:
        counts[label] = 0
        continue

    query = f"MATCH (n:{label}) RETURN count(n) AS cnt"
    result = run_query(query)

    counts[label] = result['cnt'].iloc[0] if not result.empty else 0

counts_df = pd.DataFrame(counts.items(), columns=['Node Label', 'Count'])
print(counts_df)

   Node Label  Count
0    Customer  20652
1  Department     11
2       Order  65752
3     Product    118
4      Region     23


In [14]:
# Relationship counts by type
rel_types = ['CONTAINS', 'IN_DEPARTMENT', 'PLACED', 'SHIPPED_TO']
rel_counts = {}
for rel in rel_types:
    result = run_query(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS cnt")
    rel_counts[rel] = result['cnt'].iloc[0] if not result.empty else 0

rel_df = pd.DataFrame(list(rel_counts.items()), columns=['Relationship', 'Count'])
print("=== Relationship Counts ===")
print(rel_df.to_string(index=False))

=== Relationship Counts ===
 Relationship  Count
     CONTAINS 159763
IN_DEPARTMENT    118
       PLACED  65752
   SHIPPED_TO  65752


In [18]:
GRAPH_NAME = "supply_chain_graph"

# Drop the projection if it already exists from a previous run
try:
    run_query(f"CALL gds.graph.drop('{GRAPH_NAME}', false)")
    print(f"Dropped existing projection: {GRAPH_NAME}")
except Exception:
    pass  # No existing projection — that's fine

# Create the projected graph
# orientation: 'UNDIRECTED' lets influence travel in both directions
project_query = f"""
CALL gds.graph.project(
  '{GRAPH_NAME}',
  ['Customer', 'Department', 'Order', 'Product' , 'Region'],
  {{
    CONTAINS: {{ orientation: 'UNDIRECTED' }},
    IN_DEPARTMENT: {{ orientation: 'UNDIRECTED' }},
    PLACED: {{ orientation: 'UNDIRECTED' }},
    SHIPPED_TO:       {{ orientation: 'UNDIRECTED' }}
  }}
)
YIELD graphName, nodeCount, relationshipCount, projectMillis
"""
result = run_query(project_query)
print(result.to_string(index=False))

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('supply_chain_graph', false)"


Dropped existing projection: supply_chain_graph
         graphName  nodeCount  relationshipCount  projectMillis
supply_chain_graph      86556             582770            200


In [19]:
# Run PageRank and write scores back to nodes as a property
pagerank_query = f"""
CALL gds.pageRank.write('{GRAPH_NAME}', {{
  dampingFactor:  0.85,
  maxIterations:  20,
  tolerance:      1e-7,
  writeProperty:  'pagerank'
}})
YIELD nodePropertiesWritten, ranIterations, didConverge, centralityDistribution
"""
pr_result = run_query(pagerank_query)
print("PageRank summary:")
print(pr_result.to_string(index=False))

PageRank summary:
 nodePropertiesWritten  ranIterations  didConverge                                                                                                                                                                                                                              centralityDistribution
                 86556             20        False {'p99': 1.1124629974365234, 'min': 0.265472412109375, 'max': 2359.3281249999995, 'p90': 0.8043460845947266, 'mean': 0.9612403199117281, 'p999': 32.01733207702637, 'p50': 0.6555271148681641, 'p95': 0.8717288970947266, 'p75': 0.7599163055419922}


### Top-Ranked Products

In [43]:
top_products = run_query("""
MATCH (p:Product)
WHERE p.pagerank IS NOT NULL
RETURN p.name                 AS product_name,
       p.pagerank             AS pagerank
ORDER BY p.pagerank DESC
LIMIT 20
""")

print("=== Top 20 Products by PageRank ===")
print(top_products.to_string(index=False))

=== Top 20 Products by PageRank ===
                                 product_name    pagerank
             perfect fitness perfect rip deck 2359.323173
      nike men's cj elite 2 td football cleat 2171.876488
         nike men's dri-fit victory golf polo 2064.348830
             o'brien men's neoprene life vest 1917.242443
    field & stream sportsman 16 gun fire safe 1749.610002
                  pelican sunstream 100 kayak 1581.013971
diamondback women's serene classic comfort bi 1413.334864
            nike men's free 5.0+ running shoe 1274.578247
under armour girls' toddler spine surge runni 1127.145484
                         fighting video games  187.590441
                               summer dresses  145.445094
                           children's heaters  139.853909
                                   web camera  125.846923
                                         toys  118.537282
                           adult dog supplies  110.437052
                                   l

### Top-Ranked Regions

In [32]:
top_regions = run_query("""
MATCH (r:Region)
WHERE r.pagerank IS NOT NULL
RETURN r.name               AS region,
       r.pagerank           AS pagerank
ORDER BY r.pagerank DESC
LIMIT 20
""")

print("=== Top 20 Regions by PageRank ===")
print(top_regions.to_string(index=False))

=== Top 20 Regions by PageRank ===
         region    pagerank
 Western Europe 1351.597253
Central America 1107.800114
 Southeast Asia  722.408137
        Oceania  682.968369
  South America  588.491893
   Eastern Asia  548.019140
     South Asia  525.221190
Northern Europe  505.899336
Southern Europe  480.926982
      Caribbean  332.000989
   West of USA   315.218831
    East of USA  274.855165
      West Asia  238.937919
     US Center   227.875645
 South of  USA   158.957232
 Eastern Europe  151.977369
    West Africa  144.174370
   North Africa  125.679939
    East Africa   72.727205
 Central Africa   65.996079


### Business Summary: PageRank Findings

**PageRank reveals the structural backbone of the supply chain and where late-delivery risk is most consequential.**

- The highest-PageRank products are **[perfect fitness perfect rip deck,nike men's cj elite 2 td football cleat,etc.]**. These products sit at the intersection of the most order flows, meaning delays affecting them ripple outward to the largest number of customers.

- Among regions, **[Western Europe, central America,Southeast Asia, etc]** score highest on PageRank. These are the destination hubs that the supply chain routes the most volume through. If any of these regions also show a high late-delivery rate, they represent a systemic bottleneck — a decision-maker should prioritise re-routing or pre-stocking inventory near these hubs.



## Community Detection — Louvain Algorithm


In [36]:
# Run Louvain community detection and write community IDs back to nodes
louvain_query = f"""
CALL gds.louvain.write('{GRAPH_NAME}', {{
  writeProperty:        'community_id',
  maxLevels:            10,
  maxIterations:        10,
  tolerance:            0.0001,
  includeIntermediateCommunities: false
}})
YIELD communityCount, modularity, ranLevels
"""
louvain_result = run_query(louvain_query)
print("Louvain summary:")
print(louvain_result.to_string(index=False))

Louvain summary:
 communityCount  modularity  ranLevels
             24     0.33153          3


### Community Size Distribution

In [45]:
# How big is each community?
community_sizes = run_query("""
MATCH (n)
WHERE n.community_id IS NOT NULL
RETURN n.community_id AS community_id,
       labels(n)[0]   AS node_label,
       count(n)        AS size
ORDER BY community_id, node_label
""")

pivot = community_sizes.pivot_table(
    index='community_id', columns='node_label', values='size', fill_value=0
)
pivot['Total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('Total', ascending=False)

print(pivot.head(20).to_string())

top_communities = pivot.head(10).index.tolist()
print(f"\nTop 10 communities (by size): {top_communities}")

node_label    Customer  Department   Order  Product  Region    Total
community_id                                                        
23092           5733.0         2.0  6429.0     12.0     4.0  12180.0
32141           3299.0         5.0  4940.0     24.0     3.0   8271.0
42775           1230.0         0.0  6197.0      7.0     1.0   7435.0
33636           1186.0         1.0  6138.0      5.0     1.0   7331.0
32818           1177.0         0.0  5978.0      5.0     0.0   7160.0
22854           1169.0         0.0  5971.0      6.0     0.0   7146.0
32252           1034.0         0.0  4983.0      4.0     0.0   6021.0
11448           1017.0         0.0  4795.0      4.0     0.0   5816.0
36487            728.0         3.0  3398.0      8.0     1.0   4138.0
34325            712.0         0.0  3360.0      3.0     0.0   4075.0
32218            688.0         0.0  2878.0      5.0     1.0   3572.0
37256            631.0         0.0  2743.0      1.0     0.0   3375.0
18905            292.0         0.0

In [40]:
community_late = run_query("""
MATCH (o:Order)
WHERE o.community_id IS NOT NULL
RETURN o.community_id              AS community_id,
       count(o)                    AS total_orders
ORDER BY total_orders DESC
LIMIT 15
""")

print("=== Late Delivery Rate by Community (Order nodes) ===")
print(community_late.to_string(index=False))

=== Late Delivery Rate by Community (Order nodes) ===
 community_id  total_orders
        23092          6429
        42775          6197
        33636          6138
        32818          5978
        22854          5971
        32252          4983
        32141          4940
        11448          4795
        36487          3398
        34325          3360
        32218          2878
        37256          2743
        18905          1095
        41249           996
        31145           963


### Business Summary: Community Detection Findings

**Community detection reveals that late deliveries are not uniformly distributed — they cluster into identifiable supply chain sub-networks with distinct characteristics.**

Louvain identified **[24] communities** with a modularity of **[ 0.33153]**, indicating a well-structured graph with meaningful clusters.